In [2]:
# importing necessary libraries
import pandas as pd
import numpy as np
import ast


In [3]:
# retrieve the data
df = pd.read_csv('../../data/raw/tmdb/tmdb_raw_2019_2026.csv')
df.head(5)

,tmdb_id,title,original_title,popularity,vote_average,vote_count,release_date,original_language,genre_ids,adult
0,299534,Avengers: Endgame,Avengers: Endgame,98.7212,8.200,28768,2019-04-24,en,"[12, 878, 28]",False
1,496243,Parasite,기생충,35.6564,8.490,21362,2019-05-30,ko,"[35, 53, 18]",False
2,466272,Once Upon a Time... in Hollywood,Once Upon a Time... in Hollywood,34.9897,7.419,15284,2019-07-24,en,"[35, 18, 53]",False
3,475557,Joker,Joker,34.4579,8.100,28367,2019-10-01,en,"[80, 53, 18]",False
4,429617,Spider-Man: Far From Home,Spider-Man: Far From Home,33.8307,7.392,17646,2019-06-28,en,"[28, 12, 878]",False


In [4]:
df.columns

Index(['tmdb_id', 'title', 'original_title', 'popularity', 'vote_average',
       'vote_count', 'release_date', 'original_language', 'genre_ids',
       'adult'],
      dtype='str')

In [5]:
df.dtypes

tmdb_id                int64
title                    str
original_title           str
popularity           float64
vote_average         float64
vote_count             int64
release_date             str
original_language        str
genre_ids                str
adult                   bool
dtype: object

## let's inspect each column and treat problems one by one

In [6]:
df.isnull().sum()

tmdb_id              0
title                1
original_title       1
popularity           0
vote_average         0
vote_count           0
release_date         0
original_language    0
genre_ids            0
adult                0
dtype: int64

In [7]:
df[df["title"].isnull()]

,tmdb_id,title,original_title,popularity,vote_average,vote_count,release_date,original_language,genre_ids,adult
79496,1697128,NaN,NaN,1.568,0.0,0,2026-05-14,en,"[18, 35]",False


In [8]:
df = df.dropna(subset=["title"])
print(f"Lignes apres suppression du titre manquant : {len(df)}")

Lignes apres suppression du titre manquant : 79999


### checking for duplicate movies (same tmdb_id appearing more than once)

In [9]:
df.duplicated(subset="tmdb_id").sum()

np.int64(732)

In [10]:
nb_avant = len(df)
df = df.drop_duplicates(subset="tmdb_id", keep="first")
print(f"Doublons supprimes : {nb_avant - len(df)}")

Doublons supprimes : 732


## release_date: converting from text to a real date, and extracting the year
for consistency with the other datasets (imdb, rotten tomatoes)

In [11]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year
df[["release_date", "release_year"]].head(10)

,release_date,release_year
0,2019-04-24,2019
1,2019-05-30,2019
2,2019-07-24,2019
3,2019-10-01,2019
4,2019-06-28,2019
5,2019-06-19,2019
6,2019-11-20,2019
7,2019-11-27,2019
8,2019-12-03,2019
9,2019-07-31,2019


In [12]:
# checking if any release_date failed to convert (would show up as NaT)
df["release_date"].isnull().sum()

np.int64(0)

genre_ids is stored as text after reading the csv (e.g. "[28, 12]") instead of a real list
we convert it back into a python list using ast.literal_eval

In [13]:
df["genre_ids"] = df["genre_ids"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])
df["genre_ids"].head(5)

0    [12, 878, 28]
1     [35, 53, 18]
2     [35, 18, 53]
3     [80, 53, 18]
4    [28, 12, 878]
Name: genre_ids, dtype: object

additional check: vote_average should always be between 0 and 10 on TMDB
if not, that would indicate corrupted data from the API

In [14]:
print("Min vote_average:", df["vote_average"].min())
print("Max vote_average:", df["vote_average"].max())
print("Any negative popularity?", (df["popularity"] < 0).any())
print("Any negative vote_count?", (df["vote_count"] < 0).any())

Min vote_average: 0.0
Max vote_average: 10.0
Any negative popularity? False
Any negative vote_count? False


checking which languages are present, to spot any anomaly (unexpected codes)

In [15]:
df["original_language"].value_counts()

original_language
en    38989
es     7162
fr     5528
pt     3811
de     2596
      ...  
ti        1
io        1
rm        1
lg        1
tt        1
Name: count, Length: 145, dtype: int64

final verification of data types before saving

In [16]:
df["tmdb_id"] = df["tmdb_id"].astype(int)
df["adult"] = df["adult"].astype(bool)
df.dtypes

tmdb_id                       int64
title                           str
original_title                  str
popularity                  float64
vote_average                float64
vote_count                    int64
release_date         datetime64[us]
original_language               str
genre_ids                    object
adult                          bool
release_year                  int32
dtype: object

In [17]:
# la colonne adult est quasi constante (False) et sans équivalent dans RT/Numbers
df = df.drop(columns=["adult"])
df.columns.tolist()

['tmdb_id',
 'title',
 'original_title',
 'popularity',
 'vote_average',
 'vote_count',
 'release_date',
 'original_language',
 'genre_ids',
 'release_year']

In [18]:
# reordering columns for better readability
df = df[[
    "tmdb_id", "title", "original_title", "release_date", "release_year",
    "genre_ids", "popularity", "vote_average", "vote_count",
    "original_language"
]]
df.head(5)

,tmdb_id,title,original_title,release_date,release_year,genre_ids,popularity,vote_average,vote_count,original_language
0,299534,Avengers: Endgame,Avengers: Endgame,2019-04-24,2019,"[12, 878, 28]",98.7212,8.200,28768,en
1,496243,Parasite,기생충,2019-05-30,2019,"[35, 53, 18]",35.6564,8.490,21362,ko
2,466272,Once Upon a Time... in Hollywood,Once Upon a Time... in Hollywood,2019-07-24,2019,"[35, 18, 53]",34.9897,7.419,15284,en
3,475557,Joker,Joker,2019-10-01,2019,"[80, 53, 18]",34.4579,8.100,28367,en
4,429617,Spider-Man: Far From Home,Spider-Man: Far From Home,2019-06-28,2019,"[28, 12, 878]",33.8307,7.392,17646,en


In [19]:
import os
os.makedirs("../../data/processed", exist_ok=True)

print(df.shape)
df.to_csv("../../data/processed/tmdb_clean.csv", index=False)
print("Saved to data/processed/tmdb_clean.csv")

(79267, 10)
Saved to data/processed/tmdb_clean.csv
